<a href="https://colab.research.google.com/github/Mayank2Panchal/Saving_The_Save_Offer/blob/main/5746060_Dissertation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_FILE = '/content/drive/MyDrive/Dissertation/Dissertation_qualtricsdata.xlsx'

In [ ]:
import os

file_path = '/content/drive/MyDrive/Dissertation/Dissertation_qualtricsdata.xlsx'

if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does NOT exist. Please ensure it is in your Google Drive and that Google Drive is mounted correctly.")

In [ ]:
import pandas as pd
import numpy as np
import sys
import warnings
from scipy import stats
import statsmodels.api as sm

warnings.filterwarnings('ignore')
np.random.seed(2026)   # makes bootstrap results reproducible

In [ ]:
SHEET     = 'Sheet0'
N_BOOT    = 3000      # bootstrap resamples for mediation
ORDER     = ['control', 'discount', 'flexibility']   # fixed display order for all tables

In [ ]:
#Printto a log file
import sys, io
if not hasattr(sys, '_real_stdout'):
    sys._real_stdout = sys.stdout
class Tee:
    def __init__(self):
        self.buf = io.StringIO()
        self.stdout = sys._real_stdout
    def write(self, s):
        self.stdout.write(s)
        self.buf.write(s)
    def flush(self):
        self.stdout.flush()
sys.stdout = Tee()
def header(title, char='='):
    print('\n' + char * 78)
    print(title)
    print(char * 78)

In [ ]:
# STEP 1 — LOAD THE RAW DATA
# Qualtrics exports have TWO header rows: row 0 is the variable name (Q26, Q29_1...)
# and row 1 contains the full question text. pandas reads row 0 as the column names
header('STEP 1 — LOADING RAW DATA')

df = pd.read_excel(DATA_FILE, sheet_name=SHEET)
d = df.iloc[1:].copy().reset_index(drop=True)      # drop the question-text row

print(f'Raw responses loaded: {len(d)}')
print(f'Columns in export:    {df.shape[1]}')

In [ ]:
# STEP 2 — SCREENING CASCADE
header('STEP 2 — SCREENING CASCADE')

steps = {'Raw responses': len(d)}

d = d[d['Q1'] == 'I consent to take part in the above study']
steps['Consented (Q1)'] = len(d)

d = d[d['Q2'] == 'Yes']
steps['UK resident (Q2)'] = len(d)

d = d[d['Q4'] == 'Yes']
steps['Held subscription in 24 months (Q4)'] = len(d)

d = d.copy()

for k, v in steps.items():
    print(f'  {k:42s} {v}')

In [ ]:
# STEP 3 — RECOVER THE EXPERIMENTAL CONDITION
# THE PROBLEM: the Qualtrics export contains NO column recording which of the
#   three cancellation screens each person saw.
# THE SOLUTION: the question "What would you most likely do?" exists in three
#   versions, each shown ONLY to its matching condition:
header('STEP 3 — RECOVERING EXPERIMENTAL CONDITION')

def get_condition(row):
    if pd.notna(row['Q26']): return 'control'
    if pd.notna(row['Q27']): return 'discount'
    if pd.notna(row['Q24']): return 'flexibility'
    return np.nan

d['cond_int'] = d.apply(get_condition, axis=1)

d['cond_val'] = d['Q12'].map({'26 months': 'high', '2 months': 'low'})

n_reached = d['cond_int'].notna().sum()
print(f'  Reached the experimental manipulation: {n_reached}')
print(f'  Abandoned before manipulation:         {d["cond_int"].isna().sum()}')
print('\n  Arm sizes:')
print('   ' + d['cond_int'].value_counts().reindex(ORDER).to_string().replace('\n', '\n   '))

In [ ]:
# STEP 4 — RECODE LIKERT TEXT INTO NUMBERS
header('STEP 4 — RECODING SCALES')

LIKERT = {
    'Strongly disagree': 1, 'Disagree': 2, 'Somewhat disagree': 3,
    'Neither agree nor disagree': 4,
    'Somewhat agree': 5, 'Agree': 6, 'Strongly agree': 7
}

likert_items = ['Q27_1','Q27_2','Q27_3','Q27_4',        # trust items + attention check
                'Q29_1','Q29_2','Q29_3','Q29_4',        # price fairness battery
                'Q30_1','Q30_2','Q30_3']                # manipulative intent + care
for c in likert_items:
    d[c + '_n'] = d[c].map(LIKERT)

numeric_items = ['Q25_1','Q26_1','Q26_2','Q28','Q31_1','Q33_1','Q35_1']
for c in numeric_items:
    d[c] = pd.to_numeric(d[c], errors='coerce')

d['Duration'] = pd.to_numeric(d['Duration (in seconds)'], errors='coerce')
print(f'  Recoded {len(likert_items)} Likert items and {len(numeric_items)} numeric items.')

In [ ]:
# STEP 5 — BUILD COMPOSITE SCALE SCORES
header('STEP 5 — BUILDING COMPOSITE SCALES')

d['TRUST'] = d[['Q27_1_n', 'Q27_2_n', 'Q27_3_n']].mean(axis=1)          # brand trust
d['EXPL']  = d[['Q29_2_n', 'Q30_1_n', 'Q30_2_n']].mean(axis=1)          # EXPLOITATION INFERENCE (the key mediator)
d['PFAIR'] = d[['Q29_1_n', 'Q29_3_n', 'Q29_4_n']].mean(axis=1)          # price fairness (respecified)
d['CARE']  = d['Q30_3_n']                                               # perceived care (single item)

# Rename outcome variables for readability
d['RESUB']    = d['Q26_1']    # resubscription intention
d['CROSSBUY'] = d['Q26_2']    # cross-buy intention
d['NPS']      = d['Q28']      # 0-10 recommendation
d['RETAIN']   = d['Q25_1']    # stated retention likelihood
d['PERCVAL']  = d['Q31_1']    # perceived customer value -> THE MODERATOR
d['EASE']     = d['Q33_1']    # ease of cancellation -> manipulation check
d['OFFERVAL'] = d['Q35_1']    # perceived money value of the offer -> manipulation check

# CODING DECISION: "I'm not sure" is set to MISSING rather than 0.
#   Rationale: it is genuinely neither staying nor leaving. Forcing it into a
#   category would invent data.
def coded_stay(row):
    choice = row['Q26'] if pd.notna(row['Q26']) else (row['Q27'] if pd.notna(row['Q27']) else row['Q24'])
    if pd.isna(choice):                 return np.nan
    if 'Confirm cancellation' in str(choice): return 0
    if 'not sure' in str(choice):       return np.nan
    return 1
d['STAY'] = d.apply(coded_stay, axis=1)

# Participants' OWN real subscription behaviour (exploratory moderators)
# Bracketed answers converted to numeric midpoints so they can be used continuously.
d['own_spend'] = d['Q6'].map({'£20': 10, '£20–£50': 35, '£51–£100': 75, 'More than £100': 125})
d['own_count'] = d['Q5'].map({'None currently': 0, '1': 1, '2-3': 2.5, '4-6': 5, '7 or more': 8})

print('  Created: TRUST, EXPL, PFAIR, CARE, STAY + renamed outcomes.')

In [ ]:
# STEP 6 — DATA QUALITY FLAGS
# THREE DIRECT ATTENTION MEASURES:
#   Q27_4  an instructed-response item ("please select Somewhat disagree" = 3)
#   Q12    comprehension check on the vignette
#   Q34    recall check on which screen they saw
#
# THE SPEED FLAG DECISION: an earlier wave flagged completions under 120 seconds.
#   We TEST that assumption below rather than applying it blindly.
header('STEP 6 — DATA QUALITY')

a = d[d['cond_int'].notna()].copy()      # ANALYSIS SAMPLE: only those who reached the experiment

a['fail_attn'] = (a['Q27_4_n'] != 3)                       # correct answer is 3 = "Somewhat disagree"
a['fail_comp'] = ~a['Q12'].isin(['2 months', '26 months'])
recall_map = {'control': 'Only a confirmation screen',
              'discount': 'A discount offer',
              'flexibility': 'Options to pause or switch plans'}
a['fail_recall'] = a.apply(lambda r: r['Q34'] != recall_map.get(r['cond_int']), axis=1)
a['fast'] = a['Duration'] < 120
a['n_flags'] = a[['fail_attn', 'fail_comp', 'fail_recall']].sum(axis=1)

print(f'  Analysis sample N = {len(a)}\n')
for c, lab in [('fail_attn', 'Failed attention check (Q27_4)'),
               ('fail_comp', 'Failed comprehension (Q12)'),
               ('fail_recall', 'Failed recall (Q34)')]:
    print(f'  {lab:36s} {int(a[c].sum()):3d}  ({a[c].mean()*100:.0f}%)')
print(f'  {"Passed all three":36s} {int((a.n_flags==0).sum()):3d}  ({(a.n_flags==0).mean()*100:.0f}%)')

# Validate the condition reconstruction against the independent recall check
correct = sum(a['Q34'] == a['cond_int'].map(recall_map))
print(f'\n  RECONSTRUCTION VALIDITY: Q34 agrees with inferred condition in '
      f'{correct}/{len(a)} cases ({correct/len(a)*100:.0f}%)')

# TEST the speed flag rather than assuming it
# INTERPRETATION: if fast respondents were careless, they should FAIL the direct
#   checks more often. Below, they fail LESS often -> the flag is invalid here
#   and is dropped.
print('\n  Testing whether speed predicts inattention (fast = under 120 seconds):')
n_fast = int(a['fast'].sum())
print(f'    Fast respondents: n = {n_fast} of {len(a)}')
_worse = 0
for c, lab in [('fail_attn', 'attention'), ('fail_comp', 'comprehension'), ('fail_recall', 'recall')]:
    ct = pd.crosstab(a['fast'], a[c])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    _, p_fisher = stats.fisher_exact(ct)          # exact test: cells are small
    f_rate, s_rate = a[a['fast']][c].mean(), a[~a['fast']][c].mean()
    if f_rate > s_rate:
        _worse += 1
    print(f'    {lab:14s} fast fail={f_rate*100:3.0f}%  slow fail={s_rate*100:3.0f}%   '
          f'chi2={chi2:.3f} p={p:.4f}  Fisher p={p_fisher:.4f}  (min expected cell={expected.min():.2f})')

# Conclusion is derived
if _worse >= 2:
    print(f'    -> Fast respondents fail MORE often on {_worse} of 3 checks, but the fast')
    print(f'       group is very small (n = {n_fast}), so this is fragile evidence.')
    print('       No duration filter is applied: the three direct attention measures are')
    print('       preferred, and the clean-subsample robustness check (Step 18) already')
    print('       excludes respondents who failed them.')
else:
    print('    -> Speed does not predict inattention here, so no duration filter is applied;')
    print('       the three direct attention measures are used instead.')

In [ ]:
# STEP 7 — SCALE RELIABILITY (CRONBACH'S ALPHA)
# WHAT ALPHA DOES: tests whether items claimed to measure one construct actually
#   hang together. Range 0-1. Convention: >=.70 acceptable, >=.80 good.
header("STEP 7 — SCALE RELIABILITY (Cronbach's alpha)")

def cronbach_alpha(X):
    """
    Formula: alpha = k/(k-1) * (1 - sum(item variances) / variance of total score)
    Logic: if items agree with each other, the total varies MORE than the sum of
    the individual parts, which pushes alpha up.
    """
    X = X.dropna()
    k = X.shape[1]
    return (k / (k - 1)) * (1 - X.var(ddof=1).sum() / X.sum(axis=1).var(ddof=1))

print(f"  Trust           (Q27_1,2,3)          alpha = {cronbach_alpha(a[['Q27_1_n','Q27_2_n','Q27_3_n']]):.3f}   good")
print(f"  EXPLOITATION    (Q29_2,Q30_1,Q30_2)  alpha = {cronbach_alpha(a[['Q29_2_n','Q30_1_n','Q30_2_n']]):.3f}   very good  <-- respecified")
print(f"  Price fairness  (Q29_1,Q29_3,Q29_4)  alpha = {cronbach_alpha(a[['Q29_1_n','Q29_3_n','Q29_4_n']]):.3f}   acceptable <-- respecified")

# Compare against the ORIGINAL design (Q29_2 reverse-scored into the fairness scale)
orig = a[['Q29_1_n', 'Q29_3_n', 'Q29_4_n']].copy()
orig['Q29_2R'] = 8 - a['Q29_2_n']       # reverse-score: 1<->7, 2<->6, etc.
print(f"  [AS DESIGNED: fairness 4-item        alpha = {cronbach_alpha(orig):.3f}]   barely passing")

# Item-total correlations: which item is the misfit?
# Each item is correlated against the AVERAGE OF THE OTHERS. A well-behaved item
# correlates strongly (>.40). A misfit does not.
print('\n  Item-total correlations within the AS-DESIGNED fairness scale:')
F = orig.dropna()
for c in F.columns:
    r = F[c].corr(F.drop(columns=[c]).mean(axis=1))
    flag = '  <-- MISFIT' if abs(r) < 0.35 else ''
    print(f'    {c:10s} r = {r:+.3f}{flag}')

In [ ]:
# STEP 8 — THE BLOCK 11 DISCOVERY (correlation matrix + PCA)
# THE CLAIM: Q29_2 ("I had been overpaying") was WRITTEN as a price-fairness item
#   but BEHAVES as a manipulation item. If true, it should correlate strongly with
#   Q30_1/Q30_2 and weakly with the other fairness items.
# WHY THIS IS THE CONTRIBUTION: two literatures (price fairness, persuasion
#   knowledge) are conventionally treated as distinct. At the cancellation moment
#   they appear to FUSE into one judgement.
header('STEP 8 — BLOCK 11: CORRELATION STRUCTURE')

cols = ['Q29_1_n', 'Q29_2_n', 'Q29_3_n', 'Q29_4_n', 'Q30_1_n', 'Q30_2_n', 'Q30_3_n']
labs = ['FAIR1', 'OVERPAY', 'FAIR3', 'FAIR4', 'PRESSURE', 'MANIP', 'CARE']
C = a[cols].corr().round(2)
C.index = labs; C.columns = labs
print(C.to_string())

print('\n  KEY READING - the overpayment item vs everything else:')
for c, l in [('Q30_1_n','PRESSURE (manipulation item)'), ('Q30_2_n','MANIP (manipulation item)'),
             ('Q29_1_n','FAIR1 (fairness item)'), ('Q29_3_n','FAIR3 (fairness item)'),
             ('Q29_4_n','FAIR4 (fairness item)')]:
    print(f'    OVERPAY vs {l:32s} r = {a["Q29_2_n"].corr(a[c]):+.3f}')
print('    -> Strong POSITIVE with manipulation items, weak with fairness items.')

# Principal Components Analysis
# WHAT PCA DOES: finds how many distinct underlying dimensions explain the items.
# "Eigenvalue > 1" is the conventional rule for retaining a component.
X = a[cols].dropna()
Xs = (X - X.mean()) / X.std()               # standardise so all items weigh equally
R = np.corrcoef(Xs.T)
w, v = np.linalg.eig(R)
idx = np.argsort(w)[::-1]; w = w[idx].real; v = v[:, idx].real
print(f'\n  PCA eigenvalues: {np.round(w,3)}')
print(f'  Components with eigenvalue > 1: {int((w>1).sum())}')
print(f'  Variance explained by first 3:  {w[:3].sum()/len(cols)*100:.1f}%')

In [ ]:
# STEP 9 — RANDOMISATION AND COVARIATE BALANCE
# PURPOSE: prove the three groups were comparable BEFORE the manipulation.
#   If they were not, later differences might be pre-existing rather than caused.
# NOTE: here a NON-SIGNIFICANT result is the GOOD outcome.
header('STEP 9 — RANDOMISATION AND BALANCE CHECKS')

obs = a['cond_int'].value_counts().reindex(ORDER)
chi2, p = stats.chisquare(obs.values)
print(f'  Arm sizes {list(obs.values)} vs uniform: chi2={chi2:.3f}, p={p:.4f}  '
      f'({"balanced" if p>.05 else "IMBALANCED"})')

# Are the two randomisations (value x intervention) independent?
sub = a[a['cond_val'].notna()]
ct = pd.crosstab(sub['cond_val'], sub['cond_int'])
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(f'\n  Value x Intervention independence:')
print('   ' + ct.to_string().replace('\n', '\n   '))
print(f'\n   chi2({dof}) = {chi2:.3f}, p = {p:.4f}  '
      f'-> {"INDEPENDENT (randomisation OK)" if p>.05 else "DEPENDENT (problem)"}')

# Baseline covariates
# Chi-square for categorical covariates, ANOVA for numeric ones.
print('\n  Baseline covariate balance (want p > .05):')
for col, lab in [('Q3','Age'), ('Q39','Gender'), ('Q5','Own subscription count'),
                 ('Q6','Own monthly spend'), ('Q8','Prior save offer received')]:
    ct2 = pd.crosstab(a[col], a['cond_int'])
    chi2, pp, dof, _ = stats.chi2_contingency(ct2)
    print(f'    {lab:26s} chi2={chi2:6.3f} df={dof} p={pp:.4f}  {"OK" if pp>.05 else "IMBALANCED"}')

In [ ]:
# STEP 10 — GENDER SENSITIVITY ANALYSIS
# WHY: gender came out imbalanced above. An imbalance is only a CONFOUND if the
#   variable actually affects the outcomes. This tests that directly, in three steps:
#     (1) does gender predict outcomes?  (2) does it interact with condition?
#     (3) do conclusions change when controlling for it?
header('STEP 10 — GENDER SENSITIVITY ANALYSIS')

a['female'] = (a['Q39'] == 'Female').astype(int)
a['D_disc'] = (a['cond_int'] == 'discount').astype(int)     # dummy: 1 if discount, else 0
a['D_flex'] = (a['cond_int'] == 'flexibility').astype(int)  # dummy: 1 if flexibility, else 0
# NOTE: control is the REFERENCE category (both dummies = 0), so all coefficients
#       are interpreted as "versus control".

for Y in ['EXPL', 'PFAIR', 'CARE', 'TRUST', 'NPS']:
    dd = a[['D_disc', 'D_flex', 'female', Y]].dropna().copy()
    dd['iD'] = dd.D_disc * dd.female       # interaction terms
    dd['iF'] = dd.D_flex * dd.female
    m_int = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc','D_flex','female','iD','iF']])).fit()
    m_cov = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc','D_flex','female']])).fit()
    print(f'  {Y:6s} gender main p={m_cov.pvalues["female"]:.4f} | '
          f'interactions p={m_int.pvalues["iD"]:.3f}/{m_int.pvalues["iF"]:.3f} | '
          f'discount effect adjusted b={m_cov.params["D_disc"]:+.3f} p={m_cov.pvalues["D_disc"]:.4f}')
print('  -> No main effects, no interactions, conclusions unchanged: imbalance is not a confound.')

In [ ]:
# STEP 11 — MANIPULATION CHECKS
# CRITICAL POINT: for the first two checks, NON-SIGNIFICANCE IS THE DESIRED RESULT.
#   * Ease must NOT differ -> proves all three flows are equally easy to exit,
#   * Offer value must NOT differ between discount and flexibility -> proves any
#     difference between them reflects the FORM of the offer, not its money size.
header('STEP 11 — MANIPULATION CHECKS')

g = {k: v for k, v in a.groupby('cond_int')}

ms = {k: g[k]['EASE'].dropna() for k in ORDER}
F, p = stats.f_oneway(*[ms[k] for k in ORDER])
print(f'  EASE OF CANCELLATION (want NO difference):')
print(f'    control={ms["control"].mean():.2f}  discount={ms["discount"].mean():.2f}  flexibility={ms["flexibility"].mean():.2f}')
print(f'    F={F:.3f}, p={p:.4f}  -> {"PASS: compliance confirmed" if p>.05 else "FAIL"}')

ms = {k: g[k]['OFFERVAL'].dropna() for k in ORDER}
t, p = stats.ttest_ind(ms['discount'], ms['flexibility'], equal_var=False)
print(f'\n  OFFER VALUE (want discount ~ flexibility):')
print(f'    discount={ms["discount"].mean():.2f}  flexibility={ms["flexibility"].mean():.2f}')
print(f'    t={t:.3f}, p={p:.4f}  -> {"PASS: economically equivalent" if p>.05 else "FAIL"}')

hi = a[a.cond_val=='high']['PERCVAL'].dropna(); lo = a[a.cond_val=='low']['PERCVAL'].dropna()
t, p = stats.ttest_ind(hi, lo, equal_var=False)
sp = np.sqrt(((len(hi)-1)*hi.var() + (len(lo)-1)*lo.var()) / (len(hi)+len(lo)-2))
print(f'\n  VALUE MANIPULATION (want a difference):')
print(f'    high={hi.mean():.2f} (n={len(hi)})  low={lo.mean():.2f} (n={len(lo)})')
print(f'    t={t:.3f}, p={p:.4f}, d={(hi.mean()-lo.mean())/sp:.3f}  -> {"PASS" if p<.05 else "FAIL - see note"}')
print('    NOTE: this check FAILED. Reading about being a long-tenured customer did')
print('          not make people FEEL valuable. Moderation is therefore tested on')
print('          MEASURED perceived value (Q31_1), making it quasi-experimental.')

print('\n  RECALL ACCURACY by condition:')
for k in ORDER:
    s = g[k]; c_ = (s['Q34'] == recall_map[k]).sum()
    print(f'    {k:12s} {c_:2d}/{len(s):2d} = {c_/len(s)*100:.0f}%')

In [ ]:
# STEP 12 — TREATMENT EFFECTS (the main event)
# THE LOGIC, IN TWO STEPS:
#   1. ANOVA asks "do the three groups differ AT ALL?"  F = between-group variance
#      divided by within-group variance, i.e. signal-to-noise.
#   2. Only then do pairwise t-tests ask "WHICH pairs differ?"
#      Doing step 2 without step 1 inflates the false-positive rate.
#
# EFFECT SIZES ARE REPORTED ALONGSIDE p BECAUSE:
#   p says WHETHER an effect exists; eta-squared and g say HOW BIG it is.
#   eta-squared: .01 small, .06 medium, .14 large
#   Hedges g:    .2 small,  .5 medium,  .8 large
header('STEP 12 — TREATMENT EFFECTS')

def hedges_g(x, y):
    """Standardised mean difference with small-sample correction."""
    x, y = x.dropna(), y.dropna()
    n1, n2 = len(x), len(y)
    sp = np.sqrt(((n1-1)*x.var(ddof=1) + (n2-1)*y.var(ddof=1)) / (n1+n2-2))
    return (x.mean() - y.mean()) / sp * (1 - 3/(4*(n1+n2) - 9))

def eta_squared(groups):
    """Proportion of TOTAL variance explained by group membership."""
    allv = np.concatenate([gg.values for gg in groups]); gm = allv.mean()
    ssb = sum(len(gg)*(gg.mean()-gm)**2 for gg in groups)
    return ssb / ((allv - gm)**2).sum()

DVS = [('STAY','Behavioural stay (proportion)'), ('EXPL','Exploitation inference (1-7)'),
       ('PFAIR','Price fairness (1-7)'), ('CARE','Perceived care (1-7)'),
       ('TRUST','Brand trust (1-7)'), ('NPS','NPS (0-10)'),
       ('RESUB','Resubscription intention (1-7)'), ('RETAIN','Retention likelihood (1-7)'),
       ('CROSSBUY','Cross-buy intention (1-7)')]

summary = []
for v, lab in DVS:
    ms = {k: g[k][v].dropna() for k in ORDER}
    F, p   = stats.f_oneway(*[ms[k] for k in ORDER])       # parametric test
    kw     = stats.kruskal(*[ms[k] for k in ORDER])        # non-parametric backup (no normality assumption)
    e2     = eta_squared([ms[k] for k in ORDER])
    print(f'\n{lab}')
    for k in ORDER:
        print(f'   {k:12s} M={ms[k].mean():5.2f}  SD={ms[k].std():5.2f}  '
              f'Mdn={ms[k].median():4.1f}  n={len(ms[k]):2d}')
    print(f'   ANOVA: F={F:6.3f}, p={p:.4f}, eta2={e2:.4f}   |   '
          f'Kruskal-Wallis: H={kw.statistic:.3f}, p={kw.pvalue:.4f}')
    for x, y in [('discount','control'), ('flexibility','control'), ('discount','flexibility')]:
        t, pp = stats.ttest_ind(ms[x], ms[y], equal_var=False)   # Welch: does not assume equal variances
        print(f'     {x:11s} vs {y:11s}: t={t:6.3f}  p={pp:.4f}  g={hedges_g(ms[x], ms[y]):+.3f}')
    summary.append((lab, ms['control'].mean(), ms['discount'].mean(), ms['flexibility'].mean(), F, p, e2))

print('\n' + '-'*78)
print('SUMMARY TABLE (sorted by significance)')
print('-'*78)
print(f'{"Outcome":32s}{"Ctrl":>7s}{"Disc":>7s}{"Flex":>7s}{"F":>8s}{"p":>9s}{"eta2":>8s}')
for r in sorted(summary, key=lambda x: x[5]):
    print(f'{r[0]:32s}{r[1]:7.2f}{r[2]:7.2f}{r[3]:7.2f}{r[4]:8.3f}{r[5]:9.4f}{r[6]:8.3f}')

In [ ]:
# STEP 13 — PROFILE TEST (do EXPL and PFAIR respond differently?)
# WHY THIS EXISTS: H3 predicted price fairness would NOT move. It does (p=.0003),
#   so that null prediction is rejected. But the discriminant claim can still be
#   defended IF the two constructs respond with DIFFERENT PATTERNS across conditions.
# HOW: standardise both, flip the sign of fairness so both point the same way,
#   stack them, and test the condition x measure INTERACTION. A significant
#   interaction means the profiles genuinely differ.
header('STEP 13 — PROFILE TEST (discriminant validity of EXPL vs PFAIR)')

import statsmodels.formula.api as smf
z = lambda s: (s - s.mean()) / s.std()
long = pd.concat([
    pd.DataFrame({'v': z(a['EXPL']),   'measure': 'EXPL',      'cond': a['cond_int']}),
    pd.DataFrame({'v': -z(a['PFAIR']), 'measure': 'PFAIR_rev', 'cond': a['cond_int']})
]).dropna()

# Temporarily store the global C to avoid name collision with patsy's C()
_old_C = None
if 'C' in globals():
    _old_C = globals()['C']
    del globals()['C']

try:
    model = smf.ols('v ~ C(cond)*C(measure)', data=long).fit()
    print(sm.stats.anova_lm(model, typ=2).round(4).to_string())
    print('\n  The INTERACTION row is the one that matters. If p < .05, the two')
    print('  constructs respond differently across conditions -> discriminant validity holds.')
    print('  (EXPL peaks under discount; PFAIR troughs under flexibility.)')
finally:
    # Restore the global C
    if _old_C is not None:
        globals()['C'] = _old_C

In [ ]:
# STEP 14 — MEDIATION ANALYSIS
# THE QUESTION: the offer changed the outcome. But HOW? What went through people's heads?
#
# THE MODEL:
#        a-path                    b-path
#   X ----------> MEDIATOR (M) ----------> Y
#   |                                      ^
#   +------------ c' (direct) -------------+
#
#   a = effect of condition on the mediator
#   b = effect of the mediator on the outcome, CONTROLLING for condition
#   indirect effect = a x b   <- the amount travelling THROUGH the mediator
#
# WHY BOOTSTRAP: a x b is a PRODUCT of two coefficients, and products do not follow
#   the neat distributions that standard formulas assume. Bootstrapping makes no
#   distributional assumption: it resamples the data thousands of times and reads
#   the confidence interval off the resulting spread.
#
# DECISION RULE: if the 95% CI EXCLUDES zero, the indirect effect is significant.
header(f'STEP 14 — MEDIATION ANALYSIS (bootstrap, {N_BOOT} resamples)')

def mediation(M, Y, data, n_boot=N_BOOT):
    dd = data[['D_disc', 'D_flex', M, Y]].dropna()
    A = sm.OLS(dd[M], sm.add_constant(dd[['D_disc','D_flex']])).fit()          # a-paths
    B = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc','D_flex', M]])).fit()       # b-path and c'
    C_ = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc','D_flex']])).fit()         # total effect c
    ind_d = A.params['D_disc'] * B.params[M]
    ind_f = A.params['D_flex'] * B.params[M]

    boot_d, boot_f = [], []
    for _ in range(n_boot):
        s = dd.sample(len(dd), replace=True)      # resample WITH replacement
        try:
            AA = sm.OLS(s[M], sm.add_constant(s[['D_disc','D_flex']])).fit()
            BB = sm.OLS(s[Y], sm.add_constant(s[['D_disc','D_flex', M]])).fit()
            boot_d.append(AA.params['D_disc'] * BB.params[M])
            boot_f.append(AA.params['D_flex'] * BB.params[M])
        except Exception:
            pass
    ci = lambda v: (np.percentile(v, 2.5), np.percentile(v, 97.5))
    return dict(n=len(dd), a_d=A.params['D_disc'], pa_d=A.pvalues['D_disc'],
                a_f=A.params['D_flex'], pa_f=A.pvalues['D_flex'],
                b=B.params[M], pb=B.pvalues[M],
                c_d=C_.params['D_disc'], c_f=C_.params['D_flex'],
                cp_d=B.params['D_disc'], cp_f=B.params['D_flex'],
                ind_d=ind_d, ci_d=ci(boot_d), ind_f=ind_f, ci_f=ci(boot_f))

significant = []
for M in ['EXPL', 'PFAIR', 'CARE']:
    for Y in ['TRUST', 'RESUB', 'NPS', 'CROSSBUY']:
        r = mediation(M, Y, a)
        sig = lambda ci: 'SIGNIFICANT' if (ci[0] > 0) == (ci[1] > 0) else 'ns'
        print(f'\n{M} -> {Y}   (n={r["n"]})')
        print(f'  a-path: discount={r["a_d"]:+.3f} (p={r["pa_d"]:.4f})   '
              f'flexibility={r["a_f"]:+.3f} (p={r["pa_f"]:.4f})')
        print(f'  b-path: {r["b"]:+.3f} (p={r["pb"]:.4f})')
        print(f'  total c: disc={r["c_d"]:+.3f}  flex={r["c_f"]:+.3f}   |   '
              f'direct c\': disc={r["cp_d"]:+.3f}  flex={r["cp_f"]:+.3f}')
        print(f'  INDIRECT discount    = {r["ind_d"]:+.3f}  95% CI [{r["ci_d"][0]:+.3f}, {r["ci_d"][1]:+.3f}]  {sig(r["ci_d"])}')
        print(f'  INDIRECT flexibility = {r["ind_f"]:+.3f}  95% CI [{r["ci_f"][0]:+.3f}, {r["ci_f"][1]:+.3f}]  {sig(r["ci_f"])}')
        if sig(r['ci_d']) == 'SIGNIFICANT': significant.append((M, Y, 'discount', r['ind_d'], r['ci_d']))
        if sig(r['ci_f']) == 'SIGNIFICANT': significant.append((M, Y, 'flexibility', r['ind_f'], r['ci_f']))
        # Flag suppression / inconsistent mediation
        if abs(r['cp_f']) > abs(r['c_f']) and r['ind_f'] * r['c_f'] < 0:
            print('  >>> INCONSISTENT MEDIATION (suppression): the direct effect is LARGER than')
            print('      the total effect because the indirect path runs the OPPOSITE way.')
            print('      The observed total UNDERSTATES the positive channel.')

print('\n' + '-'*78)
print('SIGNIFICANT INDIRECT EFFECTS')
print('-'*78)
for M, Y, arm, ind, ci in significant:
    print(f'  {arm:12s} {M:6s} -> {Y:9s}  indirect={ind:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]')
print(f'\n  Total: {len(significant)} significant indirect paths')

In [ ]:
# STEP 15 — MODERATION
# THE QUESTION: does the offer work the SAME for everyone, or differently for
#   different customers?
# HOW TESTED: add an INTERACTION TERM (condition x moderator) to a regression.
#   A significant interaction means the effect genuinely varies by moderator level.
header('STEP 15 — MODERATION ANALYSIS')

results = []
for MOD, mlab in [('PERCVAL','Perceived customer value (Q31_1)'),
                  ('own_spend','Own monthly spend (Q6)'),
                  ('own_count','Own subscription count (Q5)')]:
    for Y in ['EXPL', 'PFAIR', 'CARE', 'TRUST', 'RESUB', 'NPS']:
        dd = a[['D_disc', 'D_flex', MOD, Y]].dropna().copy()
        if len(dd) < 30: continue
        dd['Mc'] = dd[MOD] - dd[MOD].mean()       # centre the moderator
        dd['iD'] = dd.D_disc * dd.Mc              # interaction: discount x moderator
        dd['iF'] = dd.D_flex * dd.Mc              # interaction: flexibility x moderator
        m = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc','D_flex','Mc','iD','iF']])).fit()
        results.append((mlab, Y, 'discount',    m.params['iD'], m.pvalues['iD'], len(dd)))
        results.append((mlab, Y, 'flexibility', m.params['iF'], m.pvalues['iF'], len(dd)))

results.sort(key=lambda x: x[4])
print(f'{"Moderator":34s}{"Outcome":10s}{"Arm":13s}{"b":>8s}{"p":>9s}')
for r in results[:12]:
    star = ' ***' if r[4] < .05 else (' *' if r[4] < .10 else '')
    print(f'{r[0]:34s}{r[1]:10s}{r[2]:13s}{r[3]:+8.3f}{r[4]:9.4f}{star}')

n_sig = sum(1 for r in results if r[4] < .05)
print(f'\n  {len(results)} interaction tests run.')
print(f'  Significant at p<.05: {n_sig}  (chance alone predicts {len(results)*0.05:.1f})')
print('  HONESTY NOTE: this ratio is suggestive, not confirmatory. Only the')
print('  pre-specified customer-value hypothesis is treated as a finding; the')
print('  remainder are labelled exploratory. (Garden of forking paths.)')

In [ ]:
# STEP 16 — JOHNSON-NEYMAN
# THE PROBLEM: the interaction says "the effect depends on perceived value".
#   But WHERE exactly does it switch from helping to harming?
header('STEP 16 — JOHNSON-NEYMAN (flexibility x perceived value -> trust)')

dd = a[['D_disc', 'D_flex', 'PERCVAL', 'TRUST']].dropna().copy()
mean_mod = dd['PERCVAL'].mean()
dd['Mc'] = dd['PERCVAL'] - mean_mod
dd['iD'] = dd.D_disc * dd.Mc
dd['iF'] = dd.D_flex * dd.Mc
m = sm.OLS(dd['TRUST'], sm.add_constant(dd[['D_disc','D_flex','Mc','iD','iF']])).fit()

print(f'  Model R2 = {m.rsquared:.3f}, F = {m.fvalue:.3f}, p = {m.f_pvalue:.5f}, n = {len(dd)}')
print(f'  Flexibility main effect: b = {m.params["D_flex"]:+.3f} (p = {m.pvalues["D_flex"]:.4f})')
print(f'  Flexibility x value:     b = {m.params["iF"]:+.3f} (p = {m.pvalues["iF"]:.4f})')

print('\n  SIMPLE SLOPES - effect of flexibility vs control at each perceived-value level:')
b_f, b_if = m.params['D_flex'], m.params['iF']
cov = m.cov_params()
for lev in range(1, 8):
    w_ = lev - mean_mod
    eff = b_f + b_if * w_
    se = np.sqrt(cov.loc['D_flex','D_flex'] + 2*w_*cov.loc['D_flex','iF'] + w_**2*cov.loc['iF','iF'])
    t_ = eff / se
    p_ = 2 * (1 - stats.t.cdf(abs(t_), m.df_resid))
    direction = 'RAISES trust' if eff > 0 else 'LOWERS trust'
    print(f'    perceived value = {lev}:  effect = {eff:+.3f}  SE={se:.3f}  '\
          f't={t_:+.2f}  p={p_:.4f}  {"[SIG] " + direction if p_ < .05 else ""}')

# Solve for the transition points
tcrit = stats.t.ppf(0.975, m.df_resid)
A_ = b_if**2 - tcrit**2 * cov.loc['iF','iF']
B_ = 2 * (b_f*b_if - tcrit**2 * cov.loc['D_flex','iF'])
C_ = b_f**2 - tcrit**2 * cov.loc['D_flex','D_flex']
disc_ = B_**2 - 4*A_*C_
if disc_ > 0:
    r1 = (-B_ + np.sqrt(disc_)) / (2*A_); r2 = (-B_ - np.sqrt(disc_)) / (2*A_)
    lo_, hi_ = sorted([r1 + mean_mod, r2 + mean_mod])
    print(f'\n  JOHNSON-NEYMAN TRANSITION POINTS: {lo_:.2f} and {hi_:.2f} on the 1-7 scale')
    print(f'    Below {lo_:.2f} -> flexibility SIGNIFICANTLY INCREASES trust')
    print(f'    Between {lo_:.2f} and {hi_:.2f} -> no significant effect')
    print(f'    Above {hi_:.2f} -> flexibility SIGNIFICANTLY DECREASES trust')
    pv = a['PERCVAL'].dropna()
    print(f'\n    {(pv > hi_).sum()}/{len(pv)} = {(pv > hi_).mean()*100:.0f}% of the sample sits above the upper threshold.')
    print('    MANAGERIAL READING: the same offer builds trust with customers who feel')
    print('    peripheral and erodes it among customers who feel valued.')

In [ ]:
# STEP 17 — MULTIPLICITY CORRECTION (Benjamini-Hochberg FDR)
# THE PROBLEM: running 9 tests at p<.05 each means roughly a 37% chance that at
#   least ONE looks significant by pure luck. Reporting 9 raw p-values overstates
#   confidence.
# HOW BH WORKS: sort p-values, rank them 1..m, and give each its own threshold
#   of (rank/m) x 0.05. Compare each p against ITS OWN threshold.
header('STEP 17 — MULTIPLICITY CORRECTION (Benjamini-Hochberg, q = .05)')

tests = []
for v, lab in DVS:
    ms = {k: g[k][v].dropna() for k in ORDER}
    F, p = stats.f_oneway(*[ms[k] for k in ORDER])
    tests.append((lab, p))
tests.sort(key=lambda x: x[1])
m_tests = len(tests)

print(f'{"Rank":6s}{"Outcome":34s}{"p":>12s}{"BH threshold":>15s}   Verdict')
survivors = []
for i, (lab, p) in enumerate(tests, 1):
    crit = i / m_tests * 0.05
    ok = p <= crit
    if ok: survivors.append(lab)
    print(f'{i:<6d}{lab:34s}{p:12.6f}{crit:15.4f}   {"PASS" if ok else "fail"}')
print(f'\n  {len(survivors)} of {m_tests} survive FDR correction:')
for s_ in survivors: print(f'    - {s_}')

In [ ]:
# STEP 18 — ROBUSTNESS: DO RESULTS HOLD ON THE CLEAN SUBSAMPLE?
# LOGIC: a real effect should SURVIVE (or strengthen) when careless respondents
#   are removed. A spurious effect typically weakens.
header('STEP 18 — ROBUSTNESS CHECK')

for lab, s in [('Full sample', a),
               ('Passed attention check', a[~a.fail_attn]),
               ('Passed all three checks', a[a.n_flags == 0])]:
    gg = {k: v for k, v in s.groupby('cond_int')}
    if len(gg) < 3: continue
    F, p = stats.f_oneway(*[gg[k]['EXPL'].dropna() for k in ORDER])
    t_, pt = stats.ttest_ind(gg['discount']['EXPL'].dropna(), gg['control']['EXPL'].dropna(), equal_var=False)
    print(f'  {lab:26s} n={len(s):3d}  EXPL: F={F:6.3f} p={p:.4f}  |  discount vs control p={pt:.4f}')

In [ ]:
# STEP 19 — NPS CATEGORIES
# WHY: NPS is conventionally reported in BANDS, not as a mean. This is the version
#   a retention manager would actually recognise.
#   0-6 = Detractor, 7-8 = Passive, 9-10 = Promoter
header('STEP 19 — NPS CATEGORIES')

a['nps_group'] = pd.cut(a['NPS'], bins=[-1, 6, 8, 10], labels=['Detractor','Passive','Promoter'])
ct = pd.crosstab(a['cond_int'], a['nps_group'])
print('  Counts:'); print('   ' + ct.to_string().replace('\n', '\n   '))
print('\n  Row percentages:')
print('   ' + (pd.crosstab(a['cond_int'], a['nps_group'], normalize='index')*100).round(0).to_string().replace('\n','\n   '))
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f'\n  chi2({dof}) = {chi2:.3f}, p = {p:.4f}')

In [ ]:
# STEP 20 — OPEN-TEXT THEMATIC ANALYSIS
# WHY THIS IS POWERFUL: Q36 was asked BEFORE any mediator scale. Nobody had been
#   prompted about fairness or manipulation yet. So these responses are SPONTANEOUS
#   and cannot have been primed by the questionnaire.
header('STEP 20 — OPEN-TEXT THEMATIC ANALYSIS (Q36)')

pattern = (r'overpay|over-pay|cheat|less price|less rate|much less|for cheap|'
           r'earlier|gimmick|tactic|manipulat|pressur|retention|marketing|green')
counts = {}
for k in ORDER:
    s = a[a.cond_int == k]['Q36'].dropna().astype(str)
    hits = s[s.str.lower().str.contains(pattern, regex=True)]
    counts[k] = (len(hits), len(s))
    print(f'  {k:12s} {len(hits):2d}/{len(s):2d} = {len(hits)/len(s)*100:4.0f}% voiced exploitation/tactic themes')

table = np.array([[counts[k][0], counts[k][1]-counts[k][0]] for k in ORDER])
chi2, p, dof, _ = stats.chi2_contingency(table)
print(f'\n  chi2({dof}) = {chi2:.3f}, p = {p:.4f}')
print('  -> Qualitative evidence converges with the quantitative mediator result.')

In [ ]:
# some additional analysis
header('STEP 20B — SUPPLEMENTARY ANALYSES CITED IN THE DISSERTATION')

# 20B.1  Prior save-offer sensitivity
print('\n20B.1  PRIOR SAVE-OFFER SENSITIVITY\n')

a['prior_disc'] = (a['Q8'] == 'Yes, a discount or money off').astype(int)
a['prior_flex'] = (a['Q8'] == 'Yes, an option to pause or change my plan').astype(int)

print(f'  {"Outcome":10s}{"disc b unadj":>14s}{"p":>9s}{"disc b adj":>13s}{"p":>9s}'
      f'{"flex b adj":>13s}{"p":>9s}')
for Y in ['EXPL', 'CARE', 'PFAIR', 'TRUST', 'STAY', 'NPS', 'RESUB']:
    dd = a[['D_disc', 'D_flex', 'prior_disc', 'prior_flex', Y]].dropna()
    m_un = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc', 'D_flex']])).fit()
    m_ad = sm.OLS(dd[Y], sm.add_constant(dd[['D_disc', 'D_flex', 'prior_disc', 'prior_flex']])).fit()
    print(f'  {Y:10s}{m_un.params["D_disc"]:+14.3f}{m_un.pvalues["D_disc"]:9.4f}'
          f'{m_ad.params["D_disc"]:+13.3f}{m_ad.pvalues["D_disc"]:9.4f}'
          f'{m_ad.params["D_flex"]:+13.3f}{m_ad.pvalues["D_flex"]:9.4f}')
print('\n  -> Conclusions are unchanged under adjustment. The one sensitivity worth')
print('     reporting is the discount-vs-control contrast on TRUST, which weakens.')

# 20B.2  Discount x perceived value on trust
print('\n20B.2  BOTH INTERACTION TERMS: condition x perceived value -> TRUST\n')

dd = a[['D_disc', 'D_flex', 'PERCVAL', 'TRUST']].dropna().copy()
dd['Mc'] = dd['PERCVAL'] - dd['PERCVAL'].mean()      # centre the moderator
dd['iD'] = dd['D_disc'] * dd['Mc']
dd['iF'] = dd['D_flex'] * dd['Mc']
m_mod = sm.OLS(dd['TRUST'], sm.add_constant(dd[['D_disc', 'D_flex', 'Mc', 'iD', 'iF']])).fit()

print(f'  Model: n = {len(dd)}, R2 = {m_mod.rsquared:.3f}, F = {m_mod.fvalue:.3f}, p = {m_mod.f_pvalue:.5f}\n')
for term, lab in [('D_disc', 'Discount main effect'), ('D_flex', 'Flexibility main effect'),
                  ('Mc', 'Perceived value main effect'),
                  ('iD', 'Discount x value  (INTERACTION)'), ('iF', 'Flexibility x value  (INTERACTION)')]:
    star = '  <-- significant' if m_mod.pvalues[term] < .05 else ''
    print(f'  {lab:36s} b = {m_mod.params[term]:+.3f}   p = {m_mod.pvalues[term]:.4f}{star}')
print('\n  -> Only the FLEXIBILITY interaction is significant: the discount\'s trust cost')
print('     is roughly uniform across the customer base, the flexibility offer\'s is not.')

# 20B.3  Trust-NPS correlation within each condition
print('\n20B.3  TRUST-NPS CORRELATION WITHIN EACH CONDITION\n')

for k in ORDER:
    s = a[a['cond_int'] == k]
    r = s['TRUST'].corr(s['NPS'])
    print(f'  {k:12s} r = {r:+.2f}   (n = {s[["TRUST","NPS"]].dropna().shape[0]})')
print('\n  -> Trust and recommendation move together in the control and discount arms')
print('     but come apart under flexibility: whatever drives willingness to')
print('     recommend that offer, it is not trust.')


# 20B.4  Correlations among composite measures
print('\n20B.4  CORRELATION MATRIX - COMPOSITE MEASURES (Appendix C)\n')

comp = ['EXPL', 'PFAIR', 'CARE', 'TRUST', 'RESUB', 'NPS', 'PERCVAL']
cm = a[comp].corr().round(2)
print('  ' + cm.to_string().replace('\n', '\n  '))
print(f'\n  -> Exploitation inference vs price fairness: r = {a["EXPL"].corr(a["PFAIR"]):+.2f}')
print('     Related but distinct, which is what the respecification in Step 7 requires.')

In [ ]:
# STEP 21 — SAVE THE CLEANED DATASET
header('STEP 21 — SAVING OUTPUT')

keep = ['cond_int','cond_val','STAY','EXPL','PFAIR','CARE','TRUST','RESUB','CROSSBUY',
        'NPS','RETAIN','PERCVAL','EASE','OFFERVAL','own_spend','own_count',
        'Q3','Q39','fail_attn','fail_comp','fail_recall','n_flags','Duration']
a[keep].to_csv('cleaned_data.csv', index=False)
print('  cleaned_data.csv     - analysis dataset with all derived variables')
print('  analysis_output.txt  - full log of this run')

header('ANALYSIS COMPLETE', '=')
print(f'  Analysis sample: N = {len(a)}')
print(f'  Arms: ' + ', '.join(f'{k}={len(g[k])}' for k in ORDER))

_expl = {k: g[k]['EXPL'].dropna() for k in ORDER}
F_expl, p_expl = stats.f_oneway(*[_expl[k] for k in ORDER])
e2_expl = eta_squared([_expl[k] for k in ORDER])

_care = {k: g[k]['CARE'].dropna() for k in ORDER}
F_care, p_care = stats.f_oneway(*[_care[k] for k in ORDER])
e2_care = eta_squared([_care[k] for k in ORDER])

_stay = {k: g[k]['STAY'].dropna() for k in ORDER}

print('\n  HEADLINE FINDINGS:')
print(f'   1. Exploitation inference: F = {F_expl:.2f}, p = {p_expl:.4f}, eta2 = {e2_expl:.3f}')
print(f'   2. Perceived care:         F = {F_care:.2f}, p = {p_care:.4f}, eta2 = {e2_care:.3f}')
print('   3. Behavioural stay: ' + ', '.join(f'{k} {_stay[k].mean()*100:.0f}%' for k in ORDER))
print(f'   4. Significant indirect (mediation) effects: {len(significant)}')
for M, Y, arm, ind, ci in significant:
    print(f'        {arm} {M} -> {Y}: {ind:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}]')
print(f'   5. Outcomes surviving Benjamini-Hochberg FDR correction: {len(survivors)} of {m_tests}')

if hasattr(sys.stdout, 'buf'):
    _log = sys.stdout.buf.getvalue()
    with open('analysis_output.txt', 'w', encoding='utf-8') as fh:
        fh.write(_log)
    sys.stdout = sys._real_stdout
    print(f'\nLog written to analysis_output.txt ({len(_log.splitlines())} lines)')
else:
    print('Log already written - re-run the logging cell to start a new one.')

In [ ]:
# Save to Drive
SAVE_DIR = '/content/drive/MyDrive/'

a[keep].to_csv(SAVE_DIR + 'cleaned_data.csv', index=False)
print('Saved to:', SAVE_DIR + 'cleaned_data.csv')

# Also trigger a browser download
from google.colab import files
files.download(SAVE_DIR + 'cleaned_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Save to Drive
SAVE_DIR = '/content/drive/MyDrive/'
import shutil
shutil.copy('analysis_output.txt', SAVE_DIR + 'analysis_output.txt')
print('Saved to:', SAVE_DIR + 'analysis_output.txt')
# Also trigger a browser download
from google.colab import files
files.download('analysis_output.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>